## Direct Preference Alignment with Direct Preference Optimization (DPO)

This notebook demonstrates example of fine-tuning a language model using Direct Preference Optimization (DPO) using SmolLM2-135M-Instruct model which has already been SFT trained, so it is compatible with DPO

### Import required modules

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig

/home/monster/Desktop/codes/llm-finetuning/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### load and format dataset

In [ ]:
# download and load dataset

dataset = load_dataset(path="trl-lib/ultrafeedback_binarized", split="train")

Generating test split: 100%|██████████| 1000/1000 [00:00<00:00, 33092.72 examples/s]


In [6]:
# View sample dataset
dataset[0]

{'chosen': [{'content': 'Use the pygame library to write a version of the classic game Snake, with a unique twist',
   'role': 'user'},
  {'content': "Sure, I'd be happy to help you write a version of the classic game Snake using the pygame library! Here's a basic outline of how we can approach this:\n\n1. First, we'll need to set up the game display and create a game object that we can use to handle the game's state.\n2. Next, we'll create the game's grid, which will be used to represent the game board. We'll need to define the size of the grid and the spaces within it.\n3. After that, we'll create the snake object, which will be used to represent the player's movement. We'll need to define the size of the snake and the speed at which it moves.\n4. We'll also need to create a food object, which will be used to represent the food that the player must collect to score points. We'll need to define the location of the food and the speed at which it moves.\n5. Once we have these objects se

dataset is in conversation format with alternating user and assistant roles. Dataset also contains both chosen and rejected samples to teach the model to align preference to the chosen sample.

### Select and load model

In [ ]:
model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    device_map="auto", 
    torch_dtype=torch.float32).to(device)

# disable attention keys and values caching for DPO training
# During DPO fine-tuning, however, caching is generally unnecessary 
# and can conflict with gradient checkpointing or consume extra GPU memory
model.config.use_cache = False

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# configure tokenizer padding token to be the same as the end of sequence token
tokenizer.pad_token = tokenizer.eos_token

# set name for the finetuned model to be saved to
output_dir = "ft-models/SmolLM2-135M-Instruct-DPO"
finetuned_model_name = "SmolLM2-135M-Instruct-DPO"
finetune_tags = ['smoll-lm', 'dpo', 'preference-alignment']


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 954.33it/s] 


### View the architecture of the model

In [10]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576, padding_idx=2)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (r

Training batches contain sequences of different lengths, so shorter sequences must be padded to match the longest sequence. Some causal language models do not define a dedicated padding token; the tokenizer padding line supplies one by reusing the existing EOS token.

The trainer’s attention mask should ensure that padded positions are ignored when calculating attention and loss. This assignment changes the tokenizer’s configuration; it does not resize the model vocabulary or create a new token.

In [ ]:
traininig_args = DPOConfig(
    # training batch size per GPU
    per_device_train_batch_size=4,
    # Number of updates steps to accumulate before performing a backward/update pass.
    # Effective batch size = per_device_train_batch_size * gradient_accumulation_steps
    gradient_accumulation_steps=4,
    # Save memory by not storing activation during forward pass
    # instead recompute them during the backward pass
    gradient_checkpointing=True,
    # learning rate
    learning_rate=1e-5,
    # learning rate schedule - 'cosine' gradually decreases the learning rate following a cosine curve
    # other options include 'linear', 'constant', 'constant_with_warmup', 'polynomial', 'cosine_with_restarts'
    lr_scheduler_type='cosine',
    # Total number of training steps to perform. If provided, overrides num_train_epochs.
    max_steps=200,
    # disable model checkpointing during training
    save_strategy='no',
    # how often to log training metrics. Set to 'steps' to log every logging_steps, or 'epoch' to log at the end of each epoch.
    logging_steps = 1,
    # Directory to save model outputs and checkpoints.
    output_dir=output_dir,
    # Number of steps for learning rate warmup. During the warmup phase, 
    # the learning rate increases linearly from 0 to the initial learning rate set in the optimizer.
    # This can help stabilize training in the early stages.
    warmup_steps=20,
    # Use bfloat16 precision for faster training
    bf16=True,
    # Disable wandb/tensorboard logging
    report_to=None,
    # Keep all columns in dataset even if not used by the model. This is useful for debugging or when you want to keep additional information in the dataset.
    remove_unused_columns=False,
    # DPO-specific temperature parameter. This controls the strength of the preference model
    # Lower values (like 0.1) make the model more confident and conservative in its preferences, while higher values (like 1.0) make it more uncertain.
    beta=0.1,
    # Maximum length of the input prompt in tokens,
    max_prompt_length=1024,
    # Maximum length of prompt's + response in tokens. This is the total length of the input and output combined.
    max_seq_length=1536,
)

TypeError: DPOConfig.__init__() got an unexpected keyword argument 'max_prompt_length'